# المرحلة الثالثة: التصنيف الهرمي

في التصنيف الهرمي نحدد المنطقة أولًا، ثم نستخدم نموذج الدولة المناسب للمنطقة المتوقعة.

```text
النص → Region LinearSVC → Country model for the predicted region
```


## لماذا نموذج دولة لكل منطقة؟

تمييز أربع دول شامية أسهل من تمييز جميع الدول العربية دفعة واحدة. كما أن بعض المناطق
تحتوي في البيانات على دولة واحدة فقط، ولذلك نستخدم تعيينًا ثابتًا بدل تدريب نموذج غير مفيد.


In [ ]:
# Path ?????? ???? ??????? ????????
from pathlib import Path
# json ?????? ????? ????????
import json
# pandas ???? ???????? ?? ????
import pandas as pd

# ???? ??? ???????
ROOT = Path.cwd()
# ???? ????? ????? ?????? ??? ???????
metrics_path = ROOT / "results" / "hierarchical_country_metrics.json"
hierarchical = json.loads(metrics_path.read_text(encoding="utf-8"))
rows = []
# ???? ????? ?? ????? ??? ?? ?? ????
for region, item in hierarchical.items():
    rows.append({
        "???????": region,
        "??? ????????": item["type"],
        "Accuracy": item.get("accuracy"),
        "Macro-F1": item.get("f1_macro"),
        "?????": ", ".join(item.get("countries", [item.get("country", "")])),
    })
# ???? ???? ????? ?????? ???????
pd.DataFrame(rows)

## بناء خريطة النماذج

الملف `models/hierarchical_country/index.json` يخبر التنبؤ هل يستخدم:

- `constant`: دولة ثابتة.
- `model`: vectorizer وLinearSVC خاصان بالمنطقة.


In [ ]:
# ???? ?????? ???? ???? ????? ?????? ??????? ??? ?????
index = json.loads(
    (ROOT / "models" / "hierarchical_country" / "index.json").read_text(encoding="utf-8")
)
# ???? ??? ???????? ?????? ??????? ??? ?????
pd.DataFrame([
    {"???????": region, **info} for region, info in index.items()
])

## قراءة النتائج

كانت أفضل نتيجة بين نماذج الدولة الإقليمية في المنطقة المغاربية، بينما كانت الخليجية
أصعب بسبب تشابه اللهجات وتفاوت عدد الأمثلة بين الدول.


In [ ]:
# ???? ?????? ??????? ???? ????? ????? ???? ????
for region, item in hierarchical.items():
    # ??????? ??? type=model ????? ???? ?? ???? ?????? ?????
    if item["type"] == "model":
        print(
            f'{region}: Accuracy={item["accuracy"]:.2%}, '
            f'Macro-F1={item["f1_macro"]:.2%}'
        )